# Implementation Showcase — Phase 2 Complete

End-to-end demonstration of the **mean-field neural network** baseline for McKean-Vlasov optimal control (Pham & Warin 2024):

1. **§1** Environment — package versions, device availability
2. **§2** Mean-field NN architectures — bin-density and cylindrical encoders, permutation invariance
3. **§3** Euler-Maruyama simulator — MKV-SDE forward simulation
4. **§4** Algorithm 1 (Global DP) — training on 1D systemic risk, comparison to analytic value (≈0.1134)
5. **§5** Algorithm 6 (Global BSDE) — BSDE adjoint training, induced policy cost
6. **§6** Alg 1 vs Alg 6 — value accuracy, wall time, peak memory
7. **§7** Checkpoints — saved model weights, load-and-verify
8. **§8** Table 28 profile — 2D validation grid (all 6 cases, both encoders, both algorithms)
9. **§9** Critical-q estimation — ABM calibration for Phase 3

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PY = ROOT / '.venv' / 'Scripts' / 'python.exe'
SCRIPT = ROOT / 'scripts' / 'train_systemic_risk_baseline.py'

# Make library importable directly in notebook cells
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print('Root:  ', ROOT)
print('Python:', PY)
print('Script:', SCRIPT)

Root: C:\Users\adoni\Desktop\Interactions\project
Python: C:\Users\adoni\Desktop\Interactions\project\.venv\Scripts\python.exe
Script: C:\Users\adoni\Desktop\Interactions\project\scripts\train_systemic_risk_baseline.py


## §1 — Environment

Verify that all required packages are installed and report the available compute device.

In [ ]:
import torch
import numpy
import scipy
import matplotlib
import tqdm

print(f"PyTorch:    {torch.__version__}")
print(f"NumPy:      {numpy.__version__}")
print(f"SciPy:      {scipy.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"tqdm:       {tqdm.__version__}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nDevice: {device}")
if device == 'cuda':
    print(f"  GPU:  {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## §2 — Mean-field NN architectures

Two encoder architectures map the empirical measure $\hat{\mu}^N = \frac{1}{N}\sum_i \delta_{X^i}$ to a fixed-size vector:

- **Bin-density encoder** — hard-histogram over a fixed grid; output dimension = `bins^state_dim`
- **Cylindrical encoder** — permutation-equivariant MLP aggregated by mean; output dimension = `latent_dim`

Both architectures are **permutation-invariant** over the particle index — verified below.

In [ ]:
import torch
from mfnn_control import EncoderConfig, TrainingConfig, build_policy, sample_initial_states_with_dim
from mfnn_control.encoders import BinDensityEncoder, CylindricalEncoder

torch.manual_seed(0)

# ── 1D states ──────────────────────────────────────────────────────────────
states_1d = torch.randn(2, 64, 1)
perm_1d   = torch.randperm(64)
states_1d_perm = states_1d[:, perm_1d, :]

enc_bin_1d = BinDensityEncoder(EncoderConfig(kind='bins', state_dim=1, bins=33))
enc_cyl_1d = CylindricalEncoder(EncoderConfig(kind='cylindrical', state_dim=1, latent_dim=16))

print("1D encoders")
print(f"  BinDensity  — output: {enc_bin_1d(states_1d).shape}, "
      f"output_dim={enc_bin_1d.output_dim}, "
      f"perm-invariant: {torch.allclose(enc_bin_1d(states_1d), enc_bin_1d(states_1d_perm), atol=1e-6)}")
print(f"  Cylindrical — output: {enc_cyl_1d(states_1d).shape}, "
      f"output_dim={enc_cyl_1d.output_dim}, "
      f"perm-invariant: {torch.allclose(enc_cyl_1d(states_1d), enc_cyl_1d(states_1d_perm), atol=1e-6)}")

# ── 2D states ──────────────────────────────────────────────────────────────
states_2d = torch.randn(2, 64, 2)
perm_2d   = torch.randperm(64)
states_2d_perm = states_2d[:, perm_2d, :]

enc_bin_2d = BinDensityEncoder(EncoderConfig(kind='bins', state_dim=2, bins=10))
print("\n2D encoders")
print(f"  BinDensity 2D — output: {enc_bin_2d(states_2d).shape}, "
      f"output_dim={enc_bin_2d.output_dim}, "
      f"perm-invariant: {torch.allclose(enc_bin_2d(states_2d), enc_bin_2d(states_2d_perm), atol=1e-6)}")

# ── Network parameter counts ───────────────────────────────────────────────
train_cfg = TrainingConfig()
policy_bin = build_policy(EncoderConfig(kind='bins', state_dim=1), train_cfg)
policy_cyl = build_policy(EncoderConfig(kind='cylindrical', state_dim=1), train_cfg)
print(f"\nMeanFieldPolicy (bins encoder)        — {sum(p.numel() for p in policy_bin.parameters()):,} parameters")
print(f"MeanFieldPolicy (cylindrical encoder) — {sum(p.numel() for p in policy_cyl.parameters()):,} parameters")

# ── Initial state sampling (cases 1-6 at both dims) ───────────────────────
print("\nInitial state shapes across cases:")
for case in ('case_1', 'case_2', 'case_3', 'case_4', 'case_5', 'case_6'):
    x1 = sample_initial_states_with_dim(case, 4, 32, 1, device='cpu', dtype=torch.float32)
    x2 = sample_initial_states_with_dim(case, 4, 32, 2, device='cpu', dtype=torch.float32)
    print(f"  {case}: 1D {tuple(x1.shape)}  2D {tuple(x2.shape)}")

## §3 — Euler-Maruyama simulator

`simulate_systemic_risk` discretises the Carmona–Fouque–Sun MKV-SDE:

$$dX^i_t = \kappa\,(\bar{X}_t - X^i_t)\,dt + \alpha^i_t\,dt + \sigma\,dW^i_t$$

Below we run the **uncontrolled** system ($\alpha \equiv 0$) and visualise sample trajectories.

In [ ]:
import matplotlib.pyplot as plt
import torch
from mfnn_control import SystemicRiskConfig
from mfnn_control.systemic_risk import simulate_systemic_risk

torch.manual_seed(7)

config = SystemicRiskConfig(horizon=0.2, steps=40, particles=50, kappa=0.6, sigma=1.0, q=0.8)

class ZeroPolicy:
    def __call__(self, time, states):
        class Out:
            actions = torch.zeros_like(states)
        return Out()

initial_states = torch.randn(3, config.particles, config.state_dim) * 0.2
result = simulate_systemic_risk(ZeroPolicy(), initial_states, config)

times = torch.linspace(0, config.horizon, config.steps + 1).numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for i, ax in enumerate(axes):
    trajs = result.states[:, i, :, 0].detach().numpy()
    ax.plot(times, trajs, alpha=0.25, lw=0.8, color='steelblue')
    ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.5)
    ax.set_title(f'Sample path {i+1}')
    ax.set_xlabel('t')
    if i == 0:
        ax.set_ylabel('$X^i_t$')
fig.suptitle(f'Uncontrolled Euler-Maruyama  ({config.particles} particles, 3 paths)', fontsize=11)
plt.tight_layout()
plt.show()

print("SimulationResult tensor shapes:")
print(f"  states:        {result.states.shape}   (steps+1, batch, particles, state_dim)")
print(f"  actions:       {result.actions.shape}")
print(f"  running costs: {result.running_costs.shape}")
print(f"  terminal cost: {result.terminal_cost.shape}")

## §4 — Algorithm 1: Global Dynamic Programming (Pham & Warin)

Algorithm 1 directly minimises the global control cost $J(\alpha)$ end-to-end via the `MeanFieldPolicy` network. We train on the 1D systemic-risk model (Case 1: $\mu_0 \sim \mathcal{N}(0,\,0.2^2)$) and compare the learned cost to the **analytical value** derived from the Riccati ODE ($\approx 0.1134$).

In [ ]:
import matplotlib.pyplot as plt
import torch
from mfnn_control import (
    EncoderConfig, SystemicRiskConfig, TrainingConfig,
    systemic_risk_value, train_pham_warin_algorithm_1,
)

torch.manual_seed(7)

config     = SystemicRiskConfig(horizon=0.2, steps=20, particles=64, kappa=0.6, sigma=1.0, q=0.8)
enc_cfg    = EncoderConfig(kind='bins', state_dim=1)
train_cfg  = TrainingConfig(iterations=300, batch_size=16, initial_case='case_1',
                            training_cases=('case_1',), seed=7)

analytic = systemic_risk_value('case_1', config)
print(f"Analytic value (Riccati ODE, case_1): {analytic:.6f}")

policy, losses = train_pham_warin_algorithm_1(config, enc_cfg, train_cfg)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(losses, lw=1.2, color='royalblue', label='Training loss  (Alg 1, bins)')
ax.axhline(analytic, color='crimson', lw=1.5, ls='--', label=f'Analytic = {analytic:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Control cost  $J(\\alpha)$')
ax.set_title('Algorithm 1 — Global DP training  (1D, case_1)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

final  = losses[-1]
err    = abs(final - analytic)
print(f"\nFinal loss:     {final:.6f}")
print(f"Analytic value: {analytic:.6f}")
print(f"Absolute error: {err:.6f}")
print(f"Within 5% tol:  {err <= 0.05 * analytic}")

## §5 — Algorithm 6: Global BSDE / Pontryagin (Pham & Warin)

Algorithm 6 learns the adjoint process $Y$ via a BSDE formulation. The network is trained to match the terminal condition $Y_T = \nabla_x g(X_T, \mu_T)$, and the optimal control is recovered via the Pontryagin stationarity condition:

$$\alpha^*_t = q\,(\bar{X}_t - X^i_t) - Y^i_t$$

We also evaluate the **induced policy cost** to compare against the analytic value.

In [ ]:
import matplotlib.pyplot as plt
import torch
from mfnn_control import (
    EncoderConfig, SystemicRiskConfig, TrainingConfig,
    evaluate_algorithm_6_policy, sample_initial_states_with_dim,
    systemic_risk_value, train_pham_warin_algorithm_6,
)

torch.manual_seed(7)

config    = SystemicRiskConfig(horizon=0.2, steps=20, particles=64, kappa=0.6, sigma=1.0, q=0.8)
enc_cfg   = EncoderConfig(kind='bins', state_dim=1)
train_cfg = TrainingConfig(iterations=300, batch_size=16, initial_case='case_1',
                           training_cases=('case_1',), seed=7)

analytic = systemic_risk_value('case_1', config)

networks, losses = train_pham_warin_algorithm_6(config, enc_cfg, train_cfg)

# Evaluate the induced control cost
eval_states   = sample_initial_states_with_dim('case_1', 32, config.particles, 1,
                                                device='cpu', dtype=torch.float32)
induced_cost  = float(evaluate_algorithm_6_policy(networks, eval_states, config).detach())

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(losses, lw=1.2, color='darkorange', label='Terminal adjoint MSE  (Alg 6, bins)')
ax.set_xlabel('Iteration')
ax.set_ylabel('Terminal adjoint MSE')
ax.set_title('Algorithm 6 — Global BSDE training  (1D, case_1)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Analytic value:      {analytic:.6f}")
print(f"Induced policy cost: {induced_cost:.6f}")
print(f"Absolute error:      {abs(induced_cost - analytic):.6f}")

## §6 — Algorithm 1 vs Algorithm 6: comparison

We compare both algorithms across value accuracy, wall-clock time, and peak memory using the training script's `--benchmark` mode (2 seeds, 200 iterations — reduced for demonstration).

In [ ]:
import json, subprocess

cmd = [
    str(PY), str(SCRIPT),
    '--benchmark',
    '--encoder', 'bins',
    '--case', 'case_1',
    '--state-dim', '1',
    '--particles', '64',
    '--steps', '20',
    '--iterations', '200',
    '--batch-size', '16',
    '--seeds', '7,9',
    '--value-tolerance', '0.05',
]
raw   = subprocess.check_output(cmd, text=True, cwd=ROOT)
bench = json.loads(raw)

alg1 = bench['algorithm_1']['aggregate']
alg6 = bench['algorithm_6']['aggregate']

metrics = [
    ('mean_final_loss',          'Mean final loss'),
    ('mean_absolute_value_error','Mean |error|'),
    ('success_rate',             'Success rate (5% tol)'),
    ('mean_wall_time_s',         'Wall time (s)'),
    ('mean_peak_memory_mb',      'Peak memory (MB)'),
]

print(f"{'Metric':<32} {'Algorithm 1':>14} {'Algorithm 6':>14}")
print('─' * 62)
for key, label in metrics:
    v1 = alg1.get(key, float('nan'))
    v6 = alg6.get(key, float('nan'))
    print(f"{label:<32} {v1:>14.4f} {v6:>14.4f}")

## §7 — Saved checkpoints

Trained policy weights are persisted to `results/checkpoints/`. We list all available checkpoints, load a saved Algorithm 1 policy, and verify it produces valid outputs.

In [ ]:
import torch
from mfnn_control import EncoderConfig, TrainingConfig
from mfnn_control.encoders import build_encoder, MeanFieldPolicy

checkpoint_dir = ROOT / 'results' / 'checkpoints'
checkpoints    = sorted(checkpoint_dir.glob('*.pt'))

print(f"Checkpoints in {checkpoint_dir.relative_to(ROOT)}:")
for cp in checkpoints:
    print(f"  {cp.name:<55}  {cp.stat().st_size / 1024:6.1f} KB")

# Load and verify an Algorithm 1 checkpoint
alg1_ckpts = [c for c in checkpoints if 'alg1' in c.name]
if alg1_ckpts:
    ckpt_path = alg1_ckpts[0]
    payload   = torch.load(ckpt_path, map_location='cpu', weights_only=False)

    enc_cfg   = EncoderConfig(**payload['encoder_config'])
    train_cfg = TrainingConfig(**payload['training_config'])
    encoder   = build_encoder(enc_cfg)
    policy    = MeanFieldPolicy(encoder=encoder, hidden_dims=train_cfg.hidden_dims)
    policy.load_state_dict(payload['policy_state_dict'])
    policy.eval()

    x = torch.randn(1, 32, 1)
    with torch.no_grad():
        out = policy(0.0, x)

    print(f"\nLoaded:       {ckpt_path.name}")
    print(f"  Algorithm:   {payload['algorithm']}")
    print(f"  Seed:        {payload['seed']}")
    print(f"  Case:        {payload['training_config']['initial_case']}")
    print(f"  Encoder:     {enc_cfg.kind}")
    print(f"  Action shape:{out.actions.shape}  (batch=1, particles=32, state_dim=1)")
    print(f"  All finite:  {out.actions.isfinite().all().item()}")

## §8 — Table 28 preparation profile (2D systemic risk)

The `--table28-profile` flag emits the full experimental grid for reproducing Table 28 from Pham & Warin: all 6 initial-distribution cases × 2 encoders × 2 algorithms × 3 seeds, in 2D. This profile defines the sweep used for the full validation run.

In [ ]:
cmd = [
    str(PY), str(SCRIPT),
    '--table28-profile',
    '--particles', '256',
    '--steps', '20',
    '--iterations', '400',
    '--batch-size', '16',
    '--seeds', '7,13,21'
]
raw = subprocess.check_output(cmd, text=True, cwd=ROOT)
profile_payload = json.loads(raw)
profile_payload

## §9 — Critical-q estimation (ABM calibration)

The `--estimate-critical-q` flag sweeps interaction strength $q$ and identifies the threshold above which the uncontrolled system produces a non-trivial cascade rate. This calibrates the ABM experiments in Phase 3.

In [ ]:
cmd = [
    str(PY), str(SCRIPT),
    '--estimate-critical-q',
    '--state-dim', '2',
    '--case', 'case_1',
    '--q-min', '0.0',
    '--q-max', '1.0',
    '--q-steps', '5',
    '--default-threshold', '-0.5',
    '--target-default-rate', '0.2',
    '--mc-paths', '64',
    '--particles', '32',
    '--steps', '8',
    '--seed', '7'
]
raw = subprocess.check_output(cmd, text=True, cwd=ROOT)
critical_q_payload = json.loads(raw)
critical_q_payload